## Class balance

In [ ]:
import pandas as pd

files = {
    '12a macro base':       'data/individual_merges/parquet_saves/macro_ind_base_me.parquet',
    '12b macro robust':     'data/individual_merges/parquet_saves/macro_ind_robust_me.parquet',
    '11a base lag1':  'data/engineered/parquet_saves/me_merged_base.parquet',
    '11b robust lag1': 'data/engineered/parquet_saves/me_merged_robust.parquet',
    '11c base lag2':  'data/engineered/parquet_saves/me_merged_base_lag2.parquet',
    '11d robust lag2': 'data/engineered/parquet_saves/me_merged_robust_lag2.parquet',
}

for name, path in files.items():
    df = pd.read_parquet(path)
    n = len(df)
    n_crisis = df['crisis'].sum()
    print(f"{name}: {n_crisis}/{n} = {n_crisis/n:.1%} crisis")

print()

for name, path in files.items():
    df = pd.read_parquet(path)
    n = len(df)
    n_crisis = df['crisis'].sum()
    non_crisis = n - n_crisis
    print(f"{name}: {non_crisis}/{n} = {non_crisis/n:.1%} non-crisis")

# NB14 — Results Visualisations

Loads results from NB11a/b/c/d, RT2a/b, NB12a/b and SHAP from NB13a/b/c/d.
Produces all thesis figures: AUROC/AUPRC comparisons, DeLong tests, SHAP importance.

**Note:** NB13b and NB13d must be rerun after the FM.AST.NFRG.CN drop fix before SHAP cells are valid.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pickle
import warnings
from pathlib import Path
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

# Micro/macro membership is decided by EXCLUSION from this explicit macro list, mirroring how
# the modelling notebooks define the blocks (NB11: `micro_cols = [c for c in df.columns if c
# not in id_cols + [target] + macro_cols]`). Do NOT reintroduce a prefix-based test: the micro
# codes 'inc_nii', 'inc_op' and 'cr_impchg' match none of the g_/l_/r_/c_/x_/p_ prefixes, so a
# prefix rule silently counts three micro features as macro - it previously did, in both the
# block aggregates and the bar colours. Union across all four specs, so one set serves 11a-11d.
MACRO_FEATURES = {
    'NGDP_RPCH', 'NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS',
    'reer_g', 'fx_g', 'FS.AST.PRVT.GD.ZS', 'FM.AST.NFRG.CN', 'nfa_g', 'credit_g',
    'credit_gap',
}


def micro_shap_cols(cols):
    """Micro 'shap_*' columns: everything that is not a known macro indicator."""
    return [c for c in cols if c[5:] not in MACRO_FEATURES]

In [ ]:
BASE = Path("..").resolve()
OUTPUT = BASE / 'outputs/figures' / 'NB14 outputs/figures'
OUTPUT.mkdir(exist_ok=True)

# Results parquets / excels — one entry per configuration
RESULT_FILES = {
    '11a_base_lag1':          BASE / 'outputs/results/11a/results_11a_final.xlsx',
    '11b_robust_lag1':        BASE / 'outputs/results/11b/results_n50_trials_final.parquet',
    '11c_base_lag2':          BASE / 'outputs/results/11c/results_n50_trials_draft4.parquet',
    '11d_robust_lag2':        BASE / 'outputs/results/11d/results_n50_trials_final.parquet',
    'rt2a_base_lag2_fixed':   BASE / 'outputs/results/rt2a_draft.xlsx',
    'rt2b_robust_lag2_fixed': BASE / 'outputs/results/rt2b_final.xlsx',
    '12a_macro_base':         BASE / 'outputs/results/12a/results_12a_final.xlsx',
    '12b_macro_robust':       BASE / 'outputs/results/12b/results_12b_final.xlsx',
}

# Proba stores for DeLong tests
PROBA_FILES = {
    '11a_base_lag1':          BASE / 'outputs/results/11a/proba_store_final.pkl',
    '11b_robust_lag1':        BASE / 'outputs/results/11b/proba_store_final.pkl',
    '11c_base_lag2':          BASE / 'outputs/results/11c/proba_store.pkl',
    '11d_robust_lag2':        BASE / 'outputs/results/11d/proba_store_final.pkl',
    'rt2a_base_lag2_fixed':   BASE / 'outputs/results/rt2a/proba_store.pkl',
    'rt2b_robust_lag2_fixed': BASE / 'outputs/results/rt2b/proba_store_final.pkl',
    '12a_macro_base':         BASE / 'outputs/results/12a/proba_store_final.pkl',
    '12b_macro_robust':       BASE / 'outputs/results/12b/proba_store_final.pkl',
}

# SHAP parquets — NB13b and 13d need rerun after FM.AST.NFRG.CN fix before these are valid
SHAP_FILES = {
    '11a': BASE / 'outputs/attribution/outputs_11a/shap_values_11a.parquet',
    '11b': BASE / 'outputs/attribution/outputs_11b/shap_values_11b.parquet',
    '11c': BASE / 'outputs/attribution/outputs_11c/shap_values_11c.parquet',
    '11d': BASE / 'outputs/attribution/outputs_11d/shap_values_11d.parquet',
}

# Human-readable labels
SPEC_LABELS = {
    '11a_base_lag1':          'Baseline / Lag-1',
    '11b_robust_lag1':        'Baseline Extended / Lag-1',
    '11c_base_lag2':          'Baseline / Lag-2',
    '11d_robust_lag2':        'Baseline Extended / Lag-2',
    'rt2a_base_lag2_fixed':   'Baseline / Lag-2 (fixed split)',
    'rt2b_robust_lag2_fixed': 'Extended / Lag-2 (fixed split)',
    '12a_macro_base':         'Macro-only / Base window',
    '12b_macro_robust':       'Macro-only / Extended window',
}

## 1. Load all results into a unified DataFrame

In [ ]:
frames = []

for spec, path in RESULT_FILES.items():
    if not path.exists():
        print(f'MISSING: {spec} — {path.name}')
        continue
    if path.suffix == '.parquet':
        df = pd.read_parquet(path)
    else:
        # Excel files may have multiple sheets; take the first
        df = pd.read_excel(path, sheet_name=0)
    df['spec'] = spec
    df['spec_label'] = SPEC_LABELS[spec]
    frames.append(df)
    print(f'Loaded {spec}: {df.shape[0]} rows, variants={df["variant"].unique()}, datasets={df["dataset"].unique()}')

results = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(results)}')
print('Columns:', results.columns.tolist())

## 2. Performance summary — mean AUROC and AUPRC

This is also captured more clearly in NB15, more for a check

In [ ]:
# Summary: mean AUROC and AUPRC by spec × variant × model × dataset
for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        print(f'{metric} not in columns — skipping')
        continue
    print(f'\n=== Mean {metric.upper()} by model × dataset (baseline_t1) ===')
    for spec in results['spec'].unique():
        sub = results[(results['spec'] == spec) & (results['variant'] == 'baseline_t1')]
        if sub.empty:
            continue
        tbl = sub.groupby(['model', 'dataset'])[metric].mean().round(3).unstack('dataset')
        print(f'\n--- {SPEC_LABELS[spec]} ---')
        print(tbl.to_string())

## 3. AUROC / AUPRC heatmaps — integrated vs micro vs macro

In [ ]:
# One heatmap per spec, showing mean AUROC across models × datasets
# Focus on the four main NB11 specs + baseline_t1 variant
MAIN_SPECS = ['11a_base_lag1', '11b_robust_lag1', '11c_base_lag2', '11d_robust_lag2']
DATASETS_ORDER = ['micro', 'macro', 'integrated']

for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    fig.suptitle(f'Mean {metric.upper()} by model and feature set (baseline_t1)', fontsize=13)

    for ax, spec in zip(axes, MAIN_SPECS):
        sub = results[(results['spec'] == spec) & (results['variant'] == 'baseline_t1')]
        if sub.empty:
            ax.set_visible(False)
            continue
        pivot = sub.groupby(['model', 'dataset'])[metric].mean().round(3).unstack('dataset')
        # Reorder columns if present
        cols = [c for c in DATASETS_ORDER if c in pivot.columns]
        pivot = pivot[cols]
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
                    vmin=0.6, vmax=1.0, linewidths=0.5, cbar=True)
        ax.set_title(SPEC_LABELS[spec], fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')

    plt.tight_layout()
    plt.savefig(OUTPUT / f'heatmap_{metric}_baseline_t1.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Same heatmaps for rt1 variant
for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    fig.suptitle(f'Mean {metric.upper()} by model and feature set (rt1 — onset year excluded)', fontsize=13)

    for ax, spec in zip(axes, MAIN_SPECS):
        sub = results[(results['spec'] == spec) & (results['variant'] == 'rt1')]
        if sub.empty:
            ax.set_visible(False)
            continue
        pivot = sub.groupby(['model', 'dataset'])[metric].mean().round(3).unstack('dataset')
        cols = [c for c in DATASETS_ORDER if c in pivot.columns]
        pivot = pivot[cols]
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
                    vmin=0.6, vmax=1.0, linewidths=0.5, cbar=True)
        ax.set_title(f'{SPEC_LABELS[spec]} — RT1', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')

    plt.tight_layout()
    plt.savefig(OUTPUT / f'heatmap_{metric}_rt1.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. H1 — Integrated vs single-level: grouped bar charts

In [ ]:
# Grouped bar chart: mean AUROC per dataset, averaged across all models
# One group per spec, showing micro / macro / integrated side by side
DATASET_COLORS = {'micro': 'steelblue', 'macro': 'coral', 'integrated': 'seagreen'}

for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    for variant in ['baseline_t1', 'rt1']:
        sub = results[(results['spec'].isin(MAIN_SPECS)) & (results['variant'] == variant)]
        if sub.empty:
            continue
        agg = sub.groupby(['spec_label', 'dataset'])[metric].mean().unstack('dataset')
        cols = [c for c in DATASETS_ORDER if c in agg.columns]
        agg = agg[cols]

        x = np.arange(len(agg))
        width = 0.25
        fig, ax = plt.subplots(figsize=(11, 5))
        for i, ds in enumerate(cols):
            bars = ax.bar(x + i*width, agg[ds], width, label=ds.capitalize(),
                          color=DATASET_COLORS[ds], alpha=0.85, edgecolor='white')
        ax.set_xticks(x + width)
        ax.set_xticklabels(agg.index, rotation=15, ha='right', fontsize=9)
        ax.set_ylabel(metric.upper())
        ax.set_ylim(0.5, 1.05)
        ax.set_title(f'Mean {metric.upper()} by feature set and specification — {variant}', fontsize=11)
        ax.legend()
        ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
        plt.tight_layout()
        plt.savefig(OUTPUT / f'h1_bars_{metric}_{variant}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 5. Temporal stability — AUROC/AUPRC over test years

In [ ]:
# Line chart: mean AUROC per test year, per dataset — for each main spec
for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    for spec in MAIN_SPECS:
        sub = results[(results['spec'] == spec) & (results['variant'] == 'baseline_t1')]
        if sub.empty:
            continue
        agg = sub.groupby(['test_year', 'dataset'])[metric].mean().unstack('dataset')
        cols = [c for c in DATASETS_ORDER if c in agg.columns]

        fig, ax = plt.subplots(figsize=(9, 4))
        for ds in cols:
            ax.plot(agg.index, agg[ds], marker='o', label=ds.capitalize(),
                    color=DATASET_COLORS[ds])
        ax.set_xlabel('Test year')
        ax.set_ylabel(metric.upper())
        ax.set_title(f'{metric.upper()} over test years — {SPEC_LABELS[spec]}', fontsize=11)
        ax.legend()
        ax.set_ylim(0.4, 1.05)
        plt.tight_layout()
        plt.savefig(OUTPUT / f'temporal_{metric}_{spec}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 6. RT1 robustness — performance delta when onset year excluded

Tests whether results are inflated by crisis onset years. A large drop suggests onset years drive performance.

In [ ]:
for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    deltas = []
    for spec in MAIN_SPECS:
        base = results[(results['spec'] == spec) & (results['variant'] == 'baseline_t1')]
        rt1  = results[(results['spec'] == spec) & (results['variant'] == 'rt1')]
        if base.empty or rt1.empty:
            continue
        b_mean = base.groupby('dataset')[metric].mean()
        r_mean = rt1.groupby('dataset')[metric].mean()
        delta = (r_mean - b_mean).rename(SPEC_LABELS[spec])
        deltas.append(delta)

    if not deltas:
        continue
    delta_df = pd.concat(deltas, axis=1).T
    cols = [c for c in DATASETS_ORDER if c in delta_df.columns]
    delta_df = delta_df[cols]

    fig, ax = plt.subplots(figsize=(10, 4))
    delta_df.plot(kind='bar', ax=ax,
                  color=[DATASET_COLORS[c] for c in cols],
                  edgecolor='white', alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_ylabel(f'RT1 − Baseline (Δ {metric.upper()})')
    ax.set_title(f'Performance change when excluding onset year and t+1 (RT1 vs baseline_t1)', fontsize=11)
    ax.set_xticklabels(delta_df.index, rotation=15, ha='right')
    ax.legend(title='Feature set')
    plt.tight_layout()
    plt.savefig(OUTPUT / f'rt1_delta_{metric}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(delta_df.round(3).to_string())

In [ ]:
# SECTION 6 ADDITION — RT1 delta per model (existing pooled figures above unchanged)
# Shows whether the onset-year performance drop is consistent across model classes.
#
# Caveat: RT1 has 2 test windows (2011, 2012); baseline_t1 has 3 (2010–2012).
# Delta compares a 2-year mean against a 3-year mean — not a paired comparison.
# Pattern is informative; do not attach formal significance.

for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    available_specs = [
        s for s in MAIN_SPECS
        if not results[(results['spec'] == s) & (results['variant'] == 'baseline_t1')].empty
        and not results[(results['spec'] == s) & (results['variant'] == 'rt1')].empty
    ]
    if not available_specs:
        print(f'No specs with both baseline_t1 and rt1 for {metric}')
        continue

    fig, axes = plt.subplots(1, len(available_specs),
                             figsize=(5 * len(available_specs), 5), sharey=True)
    if len(available_specs) == 1:
        axes = [axes]
    fig.suptitle(
        f'Crisis exclusion − Baseline Δ {metric.upper()} per model',
        #'(RT1 = 2 test windows vs baseline = 3 — means not paired; visual pattern only)',
        fontsize=11
    )
    for ax, spec in zip(axes, available_specs):
        sub_base = results[(results['spec'] == spec) & (results['variant'] == 'baseline_t1')]
        sub_rt1  = results[(results['spec'] == spec) & (results['variant'] == 'rt1')]
        models_list = sorted(sub_base['model'].unique())
        x     = np.arange(len(models_list))
        width = 0.25
        for i, ds in enumerate(['micro', 'macro', 'integrated']):
            b = sub_base[sub_base['dataset'] == ds].groupby('model')[metric].mean().reindex(models_list)
            r = sub_rt1[sub_rt1['dataset']  == ds].groupby('model')[metric].mean().reindex(models_list)
            ax.bar(x + i * width, (r - b).fillna(0), width,
                   label=ds.capitalize(), color=DATASET_COLORS[ds], alpha=0.85, edgecolor='white')
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xticks(x + width)
        ax.set_xticklabels(models_list, rotation=40, ha='right', fontsize=8)
        ax.set_title(SPEC_LABELS[spec], fontsize=9)
        if ax is axes[0]:
            ax.set_ylabel(f'Δ {metric.upper()} (Crisis exclusion − baseline)')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'rt1_delta_{metric}_per_model.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. DeLong test — statistical significance of performance differences

Tests whether integrated vs micro / integrated vs macro AUROC differences are statistically significant.
The DeLong test accounts for correlation between AUROCs computed on the same test observations.

In [ ]:
def delong_test(y_true, y_proba1, y_proba2):
    """
    DeLong et al. (1988) test for comparing two correlated AUROC values.
    Returns: z-statistic, p-value, auroc1, auroc2.
    Placement-value approach: exact for finite samples.
    """
    y_true = np.array(y_true)
    pos_mask = y_true == 1
    n1 = pos_mask.sum()   # number of positives
    n0 = (~pos_mask).sum()  # number of negatives

    if n1 == 0 or n0 == 0:
        return np.nan, np.nan, np.nan, np.nan

    def placement(y_true, y_proba):
        """V10 = mean P(score_pos > score_neg) per positive; V01 per negative."""
        pos = y_proba[pos_mask]
        neg = y_proba[~pos_mask]
        V10 = np.array([np.mean(p > neg) + 0.5 * np.mean(p == neg) for p in pos])
        V01 = np.array([np.mean(n < pos) + 0.5 * np.mean(n == pos) for n in neg])
        return V10, V01

    V10_1, V01_1 = placement(y_true, np.array(y_proba1))
    V10_2, V01_2 = placement(y_true, np.array(y_proba2))

    auroc1, auroc2 = V10_1.mean(), V10_2.mean()

    # Covariance matrix components
    S10 = np.cov(np.vstack([V10_1, V10_2]))
    S01 = np.cov(np.vstack([V01_1, V01_2]))

    var_diff = (S10[0,0] + S10[1,1] - 2*S10[0,1]) / n1 + \
               (S01[0,0] + S01[1,1] - 2*S01[0,1]) / n0

    if var_diff <= 0:
        return 0.0, 1.0, auroc1, auroc2

    z = (auroc1 - auroc2) / np.sqrt(var_diff)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p, auroc1, auroc2

In [ ]:
# Run DeLong tests: integrated vs micro, integrated vs macro
# Aggregate p-values across test years using Fisher's method
delong_records = []

for spec, proba_path in PROBA_FILES.items():
    if not proba_path.exists():
        print(f'Proba store missing: {spec}')
        continue
    with open(proba_path, 'rb') as f:
        proba_store = pickle.load(f)

    models = set(k[0] for k in proba_store)
    test_years = sorted(set(k[2] for k in proba_store))
    variants = set(k[3] for k in proba_store)

    for variant in variants:
        for model in models:
            for test_year in test_years:
                key_int  = (model, 'integrated', test_year, variant)
                key_mic  = (model, 'micro',      test_year, variant)
                key_mac  = (model, 'macro',       test_year, variant)

                if key_int not in proba_store:
                    continue
                y_true, y_int = proba_store[key_int]

                for cmp_label, key_cmp in [('integrated_vs_micro', key_mic),
                                            ('integrated_vs_macro', key_mac)]:
                    if key_cmp not in proba_store:
                        continue
                    _, y_cmp = proba_store[key_cmp]
                    z, p, auroc_int, auroc_cmp = delong_test(y_true, y_int, y_cmp)
                    delong_records.append({
                        'spec': spec, 'variant': variant, 'model': model,
                        'test_year': test_year, 'comparison': cmp_label,
                        'z': z, 'p_value': p,
                        'auroc_integrated': auroc_int, 'auroc_comparison': auroc_cmp,
                        'delta_auroc': auroc_int - auroc_cmp,
                    })

delong_df = pd.DataFrame(delong_records)
print(f'DeLong records: {len(delong_df)}')
delong_df.to_parquet(OUTPUT / 'delong_results.parquet', index=False)
print('Saved to outputs/figures/delong_results.parquet')

In [ ]:
# Summary: mean delta AUROC + share of significant tests (p < 0.05) per spec × comparison
if not delong_df.empty:
    summary = delong_df.groupby(['spec', 'comparison', 'variant']).agg(
        mean_delta   = ('delta_auroc', 'mean'),
        pct_sig_05   = ('p_value', lambda x: (x < 0.05).mean()),
        pct_sig_10   = ('p_value', lambda x: (x < 0.10).mean()),
        n_tests      = ('p_value', 'count'),
    ).round(3)

    print('DeLong summary (mean delta AUROC + % significant tests):')
    print(summary.to_string())

    # Heatmap of mean delta AUROC: integrated − comparison
    for variant in ['baseline_t1', 'rt1']:
        sub = delong_df[delong_df['variant'] == variant]
        if sub.empty:
            continue
        pivot = sub.groupby(['spec', 'comparison'])['delta_auroc'].mean().unstack('comparison').round(3)
        pivot.index = [SPEC_LABELS.get(s, s) for s in pivot.index]
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
                    linewidths=0.5, ax=ax)
        ax.set_title(f'DeLong: mean AUROC delta (integrated − comparison) — {variant}', fontsize=11)
        ax.set_xlabel('Comparison')
        ax.set_ylabel('')
        plt.tight_layout()
        plt.savefig(OUTPUT / f'delong_delta_heatmap_{variant}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 8. Macro-only models (NB12) vs integrated

Fair comparison requires matching the time window. NB12 uses the full macro window (1999-2017 / 1995-2017).
Integrated models are restricted to 2004-2017. Note this when interpreting.

In [ ]:
MACRO_SPECS = ['12a_macro_base', '12b_macro_robust']

# Paired comparison: macro-only (NB12) vs integrated from corresponding NB11
MACRO_TO_INTEGRATED = {
    '12a_macro_base':   '11a_base_lag1',
    '12b_macro_robust': '11b_robust_lag1',
}

for metric in ['auroc', 'auprc']:
    if metric not in results.columns:
        continue
    rows = []
    for macro_spec, int_spec in MACRO_TO_INTEGRATED.items():
        macro_sub = results[(results['spec'] == macro_spec) & (results['variant'] == 'macro_standalone')]
        int_sub   = results[(results['spec'] == int_spec)   & (results['variant'] == 'baseline_t1') &
                             (results['dataset'] == 'integrated')]
        if macro_sub.empty or int_sub.empty:
            continue
        macro_mean = macro_sub.groupby('model')[metric].mean()
        int_mean   = int_sub.groupby('model')[metric].mean()
        comp = pd.DataFrame({'macro_only': macro_mean, 'integrated': int_mean}).dropna()
        comp['spec'] = f'{SPEC_LABELS[macro_spec]} vs {SPEC_LABELS[int_spec]}'
        rows.append(comp)

    if not rows:
        continue
    comp_df = pd.concat(rows)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(comp_df['macro_only'].dropna()))
    width = 0.35
    models = comp_df.index.get_level_values('model').unique() if 'model' in comp_df.index.names else comp_df.index.unique()

    for i, (spec_name, grp) in enumerate(comp_df.groupby('spec')):
        offset = i * (len(grp) + 1)
        pos = np.arange(len(grp)) + offset
        ax.bar(pos - width/2, grp['macro_only'], width, label=f'Macro-only ({spec_name.split(" vs ")[0]})',
               color='coral', alpha=0.85)
        ax.bar(pos + width/2, grp['integrated'], width, label=f'Integrated ({spec_name.split(" vs ")[1]})',
               color='seagreen', alpha=0.85)
        ax.set_xticks(pos)
        ax.set_xticklabels(grp.index, rotation=30, ha='right')

    ax.set_ylabel(metric.upper())
    ax.set_title(f'Macro-only (NB12) vs Integrated (NB11) — {metric.upper()}\n(Note: different time windows)', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(0.5, 1.05)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'macro_vs_integrated_{metric}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. SHAP — Global feature importance

In [ ]:
SHAP_LABELS = {
    '11a': 'Baseline / Lag-1',
    '11b': 'Baseline Extended / Lag-1',
    '11c': 'Baseline / Lag-2',
    '11d': 'Baseline Extended / Lag-2',
}

for nb, shap_path in SHAP_FILES.items():
    if not shap_path.exists():
        print(f'SHAP file missing: {nb} — skipping')
        continue

    shap_df = pd.read_parquet(shap_path)
    shap_int = shap_df[shap_df['feature_set'] == 'integrated']
    shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]

    for variant in ['baseline_t1', 'rt1']:
        sub = shap_int[shap_int['variant'] == variant]
        if sub.empty:
            continue
        mean_abs = sub[shap_cols].abs().mean().sort_values(ascending=False)
        mean_abs.index = [c[5:] for c in mean_abs.index]

        # Membership by exclusion from MACRO_FEATURES, not by prefix: 'inc_nii',
        # 'inc_op' and 'cr_impchg' match no micro prefix and were drawn as macro.
        colours = ['steelblue' if f not in MACRO_FEATURES else 'coral'
                   for f in mean_abs.index]

        fig, ax = plt.subplots(figsize=(10, max(5, len(mean_abs)*0.3)))
        ax.barh(mean_abs.index[::-1], mean_abs.values[::-1], color=colours[::-1])
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title(f'Global feature importance — [NB{nb}, {variant}]', fontsize=11)
        # Legend
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color='steelblue', label='Micro'),
                            Patch(color='coral',     label='Macro')], fontsize=9)
        plt.tight_layout()
        plt.savefig(OUTPUT / f'shap_global_{nb}_{variant}.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# SECTION 9 ADDITIONS (existing per-spec figures above unchanged)

# --- 9A: Primary thesis figure — 11a baseline_t1, pooled across all models ---
_nb = '11a'
_path = SHAP_FILES.get(_nb)
if _path and _path.exists():
    _shap  = pd.read_parquet(_path)
    _si    = _shap[(_shap['feature_set'] == 'integrated') & (_shap['variant'] == 'baseline_t1')]
    _scols = [c for c in _si.columns if c.startswith('shap_')]
    _ma    = _si[_scols].abs().mean().sort_values(ascending=False)
    _ma.index = [c[5:] for c in _ma.index]
    _colours  = ['#1f77b4' if f not in MACRO_FEATURES
                 else '#ff7f0e' for f in _ma.index]
    from matplotlib.patches import Patch
    fig, ax = plt.subplots(figsize=(9, max(5, len(_ma) * 0.38)))
    ax.barh(_ma.index[::-1], _ma.values[::-1], color=_colours[::-1])
    ax.set_xlabel('Mean |SHAP value|  (pooled across all models and test years)')
    ax.set_title('Global feature importance — Baseline / Lag-1  [NB11a, baseline_t1]', fontsize=11)
    ax.legend(handles=[Patch(color='#1f77b4', label='Micro'),
                        Patch(color='#ff7f0e', label='Macro')], fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'shap_global_11a_primary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'SHAP file missing: {_nb}')

# --- 9B: Cross-spec appendix heatmap — top-15 features (11a ranking) × available specs ---
_ref_path = SHAP_FILES.get('11a')
if _ref_path and _ref_path.exists():
    _ref   = pd.read_parquet(_ref_path)
    _ref_i = _ref[(_ref['feature_set'] == 'integrated') & (_ref['variant'] == 'baseline_t1')]
    _rc    = [c for c in _ref_i.columns if c.startswith('shap_')]
    _rank  = _ref_i[_rc].abs().mean().sort_values(ascending=False)
    _rank.index = [c[5:] for c in _rank.index]
    _top15 = _rank.head(15).index.tolist()

    _spec_means = {}
    for _nb_k, _sp in SHAP_FILES.items():
        if not _sp.exists():
            continue
        _s    = pd.read_parquet(_sp)
        _si2  = _s[(_s['feature_set'] == 'integrated') & (_s['variant'] == 'baseline_t1')]
        _cmap = {c[5:]: c for c in _si2.columns if c.startswith('shap_')}
        _spec_means[SHAP_LABELS[_nb_k]] = {f: _si2[_cmap[f]].abs().mean()
                                            for f in _top15 if f in _cmap}

    if _spec_means:
        _heat  = pd.DataFrame(_spec_means).loc[_top15]
        _rcol  = ['#1f77b4' if f not in MACRO_FEATURES
                  else '#ff7f0e' for f in _top15]
        fig, ax = plt.subplots(figsize=(max(6, len(_spec_means) * 2 + 2), 8))
        sns.heatmap(_heat, annot=True, fmt='.3f', cmap='YlOrRd',
                    linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean |SHAP|'})
        ax.set_title(
            'Global feature importance — top-15 features (11a ranking) across specifications\n'
            'baseline_t1  |  blue label = micro feature, orange = macro',
            fontsize=10
        )
        for tick, c in zip(ax.get_yticklabels(), _rcol):
            tick.set_color(c)
        plt.tight_layout()
        plt.savefig(OUTPUT / 'shap_global_crossspec_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Cross-spec heatmap: 11a SHAP missing')

## 10. SHAP — Micro vs macro aggregate contribution

In [ ]:
split_records = []

for nb, shap_path in SHAP_FILES.items():
    if not shap_path.exists():
        continue
    shap_df = pd.read_parquet(shap_path)
    shap_int = shap_df[shap_df['feature_set'] == 'integrated']
    shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]

    micro_cols = micro_shap_cols(shap_cols)
    macro_cols = [c for c in shap_cols if c not in micro_cols]

    for variant in shap_int['variant'].unique():
        for model in shap_int['model_name'].unique():
            sub = shap_int[(shap_int['variant'] == variant) & (shap_int['model_name'] == model)]
            split_records.append({
                'nb': nb, 'spec': SHAP_LABELS[nb], 'variant': variant, 'model': model,
                'micro': sub[micro_cols].abs().mean().sum(),
                'macro': sub[macro_cols].abs().mean().sum(),
            })

split_df = pd.DataFrame(split_records)

# Grouped bar: micro vs macro contribution per spec × model
for variant in ['baseline_t1', 'rt1']:
    sub = split_df[split_df['variant'] == variant]
    if sub.empty:
        continue
    pivot = sub.groupby(['spec', 'model'])[['micro', 'macro']].mean().unstack('model')

    fig, ax = plt.subplots(figsize=(12, 5))
    grp = sub.groupby(['spec', 'model'])[['micro', 'macro']].mean().reset_index()
    grp_pivot = grp.pivot(index='spec', columns='model', values='micro')

    # Simpler: stacked bar per model showing micro/macro split
    agg = sub.groupby(['spec', 'model'])[['micro', 'macro']].mean().reset_index()
    models_list = agg['model'].unique()
    x = np.arange(len(agg['spec'].unique()))
    width = 0.8 / len(models_list)

    for i, model in enumerate(models_list):
        m = agg[agg['model'] == model]
        total = m['micro'] + m['macro']
        ax.bar(x + i*width, m['micro']/total, width, label=f'{model} (micro)',
               color='steelblue', alpha=0.7 - i*0.05)

    ax.set_xticks(x + width * len(models_list)/2)
    ax.set_xticklabels(agg['spec'].unique(), rotation=15, ha='right')
    ax.set_ylabel('Micro share of total SHAP')
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--', label='50% line')
    ax.set_title(f'Micro share of total SHAP contribution — {variant}', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'shap_micro_share_{variant}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# SECTION 10 ADDITIONS (existing figures above unchanged)

# --- 10A: Primary RQ1 figure — horizontal stacked bar, 11a baseline_t1 ---
# One bar per model; split into micro% / macro% of total mean |SHAP|.
_nb = '11a'
_path = SHAP_FILES.get(_nb)
if _path and _path.exists():
    _shap  = pd.read_parquet(_path)
    _si    = _shap[(_shap['feature_set'] == 'integrated') & (_shap['variant'] == 'baseline_t1')]
    _scols = [c for c in _si.columns if c.startswith('shap_')]
    _mcols = micro_shap_cols(_scols)
    _mkcols= [c for c in _scols if c not in _mcols]
    _rows  = []
    for _model in sorted(_si['model_name'].unique()):
        _sub = _si[_si['model_name'] == _model]
        _mi  = _sub[_mcols].abs().mean().sum()
        _ma  = _sub[_mkcols].abs().mean().sum()
        _tot = _mi + _ma
        _rows.append({'model': _model, 'Micro': _mi / _tot, 'Macro': _ma / _tot})
    _sdf = pd.DataFrame(_rows).set_index('model')

    fig, ax = plt.subplots(figsize=(7, max(4, len(_sdf) * 0.65)))
    _left = np.zeros(len(_sdf))
    for _col, _col_c in [('Micro', '#1f77b4'), ('Macro', '#ff7f0e')]:
        ax.barh(_sdf.index, _sdf[_col], left=_left, color=_col_c, label=_col, alpha=0.85)
        _left += _sdf[_col].values
    ax.axvline(0.5, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Share of total mean |SHAP|')
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    #ax.set_title(
    #    'Relative micro vs macro contribution — integrated configuration', fontweight='bold',
    #    fontsize=10
    #)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'shap_micro_macro_stacked_11a.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(_sdf.round(3).to_string())
else:
    print(f'SHAP file missing: {_nb}')

# --- 10B: Cross-spec robustness view — pooled across models, one bar per spec ---
_cross_rows = []
for _nb_k, _sp in SHAP_FILES.items():
    if not _sp.exists():
        continue
    _s    = pd.read_parquet(_sp)
    _si2  = _s[(_s['feature_set'] == 'integrated') & (_s['variant'] == 'baseline_t1')]
    _sc   = [c for c in _si2.columns if c.startswith('shap_')]
    _mc   = micro_shap_cols(_sc)
    _mkc  = [c for c in _sc if c not in _mc]
    _mi   = _si2[_mc].abs().mean().sum()
    _ma   = _si2[_mkc].abs().mean().sum()
    _tot  = _mi + _ma
    _cross_rows.append({'spec': SHAP_LABELS[_nb_k],
                        'Micro': _mi / _tot, 'Macro': _ma / _tot})

if _cross_rows:
    _cdf = pd.DataFrame(_cross_rows).set_index('spec')
    fig, ax = plt.subplots(figsize=(max(5, len(_cdf) * 1.8), 4))
    _left2 = np.zeros(len(_cdf))
    for _col, _col_c in [('Micro', '#1f77b4'), ('Macro', '#ff7f0e')]:
        ax.bar(_cdf.index, _cdf[_col], bottom=_left2, color=_col_c, label=_col, alpha=0.85)
        _left2 += _cdf[_col].values
    ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Share of total mean |SHAP|')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    #ax.set_title('Micro vs macro SHAP share across specifications',
    #             fontsize=10, fontweight='bold')
    ax.set_xticklabels(_cdf.index, rotation=15, ha='right')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'shap_micro_macro_stacked_crossspec.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 10C: baseline_t1 vs RT1 comparison — does micro/macro SHAP share shift when onset year excluded? ---
# Two panels, same stacked bar design as 10A. sharey=True so model labels appear once (left panel only).

_nb = '11a'
_path = SHAP_FILES.get(_nb)

_MODEL_LABELS_10C = {
    'logistic_regression': 'Logistic Regression',
    'decision_tree':       'Decision Tree',
    'random_forest':       'Random Forest',
    'xgboost':             'XGBoost',
    'lightgbm':            'LightGBM',
    'mlp':                 'MLP',
}
_MODEL_ORDER_10C = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm', 'mlp']

if _path and _path.exists():
    _shap     = pd.read_parquet(_path)
    _shap_int = _shap[_shap['feature_set'] == 'integrated']
    _scols    = [c for c in _shap_int.columns if c.startswith('shap_')]
    _mcols    = micro_shap_cols(_scols)
    _mkcols   = [c for c in _scols if c not in _mcols]

    _variant_data = {}
    for _variant in ['baseline_t1', 'rt1']:
        _sub = _shap_int[_shap_int['variant'] == _variant]
        if _sub.empty:
            continue
        _rows = []
        for _model in _MODEL_ORDER_10C:
            _ms = _sub[_sub['model_name'] == _model]
            if _ms.empty:
                continue
            _mi  = _ms[_mcols].abs().mean().sum()
            _ma  = _ms[_mkcols].abs().mean().sum()
            _tot = _mi + _ma
            _rows.append({'model': _MODEL_LABELS_10C.get(_model, _model),
                          'Micro': _mi / _tot, 'Macro': _ma / _tot})
        _variant_data[_variant] = pd.DataFrame(_rows).set_index('model')

    if len(_variant_data) == 2:
        _titles = {
            'baseline_t1': 'Baseline training  [baseline_t1]',
            'rt1':         'Onset year excluded  [RT1]',
        }
        fig, axes = plt.subplots(1, 2,
                                  figsize=(12, max(4, len(_MODEL_ORDER_10C) * 0.65)),
                                  sharey=True)
        for ax, _variant in zip(axes, ['baseline_t1', 'rt1']):
            _df   = _variant_data[_variant]
            _left = np.zeros(len(_df))
            for _col, _col_c in [('Micro', '#1f77b4'), ('Macro', '#ff7f0e')]:
                ax.barh(_df.index, _df[_col], left=_left,
                        color=_col_c, label=_col, alpha=0.85)
                _left += _df[_col].values
            ax.axvline(0.5, color='black', linewidth=0.8, linestyle='--')
            ax.set_xlim(0, 1)
            ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
            ax.set_xlabel('Share of total mean |SHAP|')
            ax.set_title(_titles[_variant], fontsize=10)
            ax.legend(fontsize=9)

        fig.suptitle(
            'Micro vs macro SHAP contribution — baseline training vs onset year excluded (RT1)'
            'NB11a  |  integrated feature set  |  pooled across test years',
            fontsize=10, y=1.02
        )
        plt.tight_layout()
        plt.savefig(OUTPUT / 'shap_micro_macro_stacked_rt1_comparison.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # Print numeric summary for reference
        for _v, _df in _variant_data.items():
            print(f'{_v}:')
            print(_df.round(3).to_string())
    else:
        print('One or both variants missing — check SHAP parquet contains both baseline_t1 and rt1')
else:
    print(f'SHAP file missing: {_nb}')

## 11. SHAP — Top feature importance over time (heatmap)

In [ ]:
for nb, shap_path in SHAP_FILES.items():
    if not shap_path.exists():
        continue
    shap_df = pd.read_parquet(shap_path)
    shap_int = shap_df[shap_df['feature_set'] == 'integrated']
    shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]

    for variant in ['baseline_t1', 'rt1']:
        sub = shap_int[shap_int['variant'] == variant]
        if sub.empty:
            continue
        # Keep only top 15 features by overall importance
        mean_abs = sub[shap_cols].abs().mean().sort_values(ascending=False)
        top_cols = mean_abs.head(15).index.tolist()

        temporal = sub.groupby('test_year')[top_cols].apply(lambda x: x.abs().mean())
        temporal.columns = [c[5:] for c in temporal.columns]

        fig, ax = plt.subplots(figsize=(12, 7))
        sns.heatmap(temporal.T, cmap='YlOrRd', ax=ax, linewidths=0.3)
        ax.set_title(f'Feature importance over test years — {SHAP_LABELS[nb]}  [NB{nb}, {variant}]', fontsize=11)
        ax.set_xlabel('Test year')
        ax.set_ylabel('Feature')
        plt.tight_layout()
        plt.savefig(OUTPUT / f'shap_temporal_{nb}_{variant}.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# SECTION 11 ADDITIONS — three condensed options for temporal SHAP heatmap
# Run all three and decide visually which reads best before committing to one.

# --- Option A: 11a only — single heatmap, baseline_t1 ---
_path_A = SHAP_FILES.get('11a')
if _path_A and _path_A.exists():
    _s    = pd.read_parquet(_path_A)
    _si   = _s[(_s['feature_set'] == 'integrated') & (_s['variant'] == 'baseline_t1')]
    _sc   = [c for c in _si.columns if c.startswith('shap_')]
    _top  = _si[_sc].abs().mean().sort_values(ascending=False).head(15).index.tolist()
    _temp = _si.groupby('test_year')[_top].apply(lambda x: x.abs().mean())
    _temp.columns = [c[5:] for c in _temp.columns]
    fig, ax = plt.subplots(figsize=(5, 7))
    sns.heatmap(_temp.T, cmap='YlOrRd', ax=ax, annot=True, fmt='.3f', linewidths=0.3)
    ax.set_title('Option A — feature importance over test years\nNB11a  Baseline / Lag-1  [baseline_t1]',
                 fontsize=10)
    ax.set_xlabel('Test year')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'shap_temporal_optionA_11a.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Option A: SHAP 11a missing')

# --- Option B: 11a + 11b side by side (base vs robust) ---
_panels_B = {}
for _nb_k, _label in [('11a', 'Baseline / Lag-1'), ('11b', 'Baseline Extended / Lag-1')]:
    _sp = SHAP_FILES.get(_nb_k)
    if _sp and _sp.exists():
        _s  = pd.read_parquet(_sp)
        _si = _s[(_s['feature_set'] == 'integrated') & (_s['variant'] == 'baseline_t1')]
        _sc = [c for c in _si.columns if c.startswith('shap_')]
        _top = _si[_sc].abs().mean().sort_values(ascending=False).head(15).index.tolist()
        _t   = _si.groupby('test_year')[_top].apply(lambda x: x.abs().mean())
        _t.columns = [c[5:] for c in _t.columns]
        _panels_B[_label] = _t

if _panels_B:
    fig, axes = plt.subplots(1, len(_panels_B), figsize=(5 * len(_panels_B), 7), sharey=False)
    if len(_panels_B) == 1:
        axes = [axes]
    for ax, (label, temp) in zip(axes, _panels_B.items()):
        sns.heatmap(temp.T, cmap='YlOrRd', ax=ax, annot=True, fmt='.3f', linewidths=0.3)
        ax.set_title(f'Option B — {label}\n[baseline_t1]', fontsize=10)
        ax.set_xlabel('Test year')
    plt.suptitle('Feature importance over test years — base vs robust', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'shap_temporal_optionB_11a_11b.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Option B: no SHAP files available')

# --- Option C: all available specs combined — features × (spec × year) ---
# Note: likely dense — view to judge readability
_all_specs_C = {}
for _nb_k, _sp in SHAP_FILES.items():
    if not _sp.exists():
        continue
    _s  = pd.read_parquet(_sp)
    _si = _s[(_s['feature_set'] == 'integrated') & (_s['variant'] == 'baseline_t1')]
    _sc = [c for c in _si.columns if c.startswith('shap_')]
    _top_ref = _si[_sc].abs().mean().sort_values(ascending=False).head(15).index.tolist()
    _t = _si.groupby('test_year')[_top_ref].apply(lambda x: x.abs().mean())
    _t.columns = [c[5:] for c in _t.columns]
    _t.index = [f'{SHAP_LABELS[_nb_k]}\n{yr}' for yr in _t.index]
    _all_specs_C[_nb_k] = _t

if _all_specs_C:
    _ref_feats  = list(list(_all_specs_C.values())[0].columns)
    _combined_C = pd.concat(list(_all_specs_C.values()), axis=0)[_ref_feats]
    fig, ax = plt.subplots(figsize=(max(8, len(_ref_feats) * 0.7),
                                    max(5, len(_combined_C) * 0.65)))
    sns.heatmap(_combined_C.T, cmap='YlOrRd', ax=ax, linewidths=0.3, annot=False)
    ax.set_title('Option C — all specs × test years  [baseline_t1]', fontsize=10)
    ax.set_xlabel('Spec / year')
    ax.set_ylabel('Feature')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'shap_temporal_optionC_all.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Option C: no SHAP files available')

## 12. H3 — Macro-conditioned AUROC split

Tests H3: does the predictive contribution of micro-level indicators change depending
on macro conditions?

Each test observation (pooled across 2010–2012) is assigned a macro regime:
- **Base specs (11a/c):** GDP growth — *recession* (NGDP_RPCH < 0) vs *expansion*
- **Robust specs (11b/d):** credit gap — *high gap* (above test-period median) vs *low gap*

Key measure: **AUROC(integrated) − AUROC(macro-only)** per regime.
This is the value that micro features add on top of the macro signal.
If H3 holds, this delta differs across regimes — micro is more (or less) useful
in one macro condition than the other.

**Index reconstruction:** test set = `df[year == test_year]` in original row order.
NB11 used `SimpleImputer` inside the training loop — no rows were dropped from the
test set — so the proba_store list order matches the merged DataFrame row order exactly.

In [ ]:
from sklearn.metrics import roc_auc_score

TEST_YEARS_H3 = [2010, 2011, 2012]

# Per-spec: which merged dataset and regime variable to use.
# Base specs lack credit_gap (not in base macro data) so fall back to GDP growth.
H3_SPECS = {
    '11a_base_lag1': {
        'proba_path':  BASE / 'outputs/results/11a/proba_store_final.pkl',
        'data_path':   BASE / 'data/engineered/parquet_saves/me_merged_base.parquet',
        'regime_col':  'NGDP_RPCH',
        'regime_type': 'gdp',
    },
    '11b_robust_lag1': {
        'proba_path':  BASE / 'outputs/results/11b/proba_store_final.pkl',
        'data_path':   BASE / 'data/engineered/parquet_saves/me_merged_robust.parquet',
        'regime_col':  'credit_gap',
        'regime_type': 'gap',
    },
    '11c_base_lag2': {
        'proba_path':  BASE / 'outputs/results/11c/proba_store.pkl',
        'data_path':   BASE / 'data/engineered/parquet_saves/me_merged_base_lag2.parquet',
        'regime_col':  'NGDP_RPCH',
        'regime_type': 'gdp',
    },
    '11d_robust_lag2': {
        'proba_path':  BASE / 'outputs/results/11d/proba_store_final.pkl',
        'data_path':   BASE / 'data/engineered/parquet_saves/me_merged_robust_lag2.parquet',
        'regime_col':  'credit_gap',
        'regime_type': 'gap',
    },
}


def build_pred_frame(proba_path, data_path, regime_col, regime_type):
    """
    Zip proba_store predictions with the original test DataFrame rows to produce
    a per-observation predictions table that includes macro regime labels.

    Works because the test set is df[year == test_year] with no row drops —
    the list order in proba_store matches the DataFrame row order exactly.
    """
    with open(proba_path, 'rb') as fh:
        ps = pickle.load(fh)

    df = pd.read_parquet(data_path)

    if regime_col not in df.columns:
        raise ValueError(f'{regime_col} not found in {data_path.name}')

    # Regime label function — NaN inputs return NaN (excluded later via dropna)
    if regime_type == 'gdp':
        def label(v):
            if pd.isna(v):
                return np.nan
            return 'recession' if v < 0 else 'expansion'
    else:
        # Median computed on test years only so it reflects the evaluation window
        med = df[df['year'].isin(TEST_YEARS_H3)][regime_col].median()
        def label(v):
            if pd.isna(v):
                return np.nan
            return 'high_gap' if v > med else 'low_gap'

    # Exclude lr_no_dummies — different feature set, not comparable to main models
    models = sorted(set(k[0] for k in ps) - {'lr_no_dummies'})

    frames = []
    for test_year in TEST_YEARS_H3:
        test = df[df['year'] == test_year].reset_index(drop=True)
        regime = test[regime_col].apply(label)

        for model in models:
            key_int = (model, 'integrated', test_year, 'baseline_t1')
            key_mic = (model, 'micro',      test_year, 'baseline_t1')
            key_mac = (model, 'macro',      test_year, 'baseline_t1')

            if not all(k in ps for k in [key_int, key_mic, key_mac]):
                continue

            y_true, p_int = ps[key_int]
            _,      p_mic = ps[key_mic]
            _,      p_mac = ps[key_mac]

            # Length must match — if this fires, the test set changed since the model ran
            assert len(y_true) == len(test), (
                f'Row mismatch: {model}/{test_year} — '
                f'proba_store has {len(y_true)} rows, df has {len(test)}'
            )

            frames.append(pd.DataFrame({
                'model':              model,
                'test_year':          test_year,
                'bank_id':            test['bank_id'].values,
                'country_iso':        test['country_iso'].values,
                'crisis':             list(y_true),
                'regime':             regime.values,
                'y_proba_integrated': list(p_int),
                'y_proba_micro':      list(p_mic),
                'y_proba_macro':      list(p_mac),
            }))

    return pd.concat(frames, ignore_index=True)

In [ ]:
MIN_POSITIVES = 10  # below this, AUROC is unreliable

h3_records = []

for spec, cfg in H3_SPECS.items():
    if not cfg['proba_path'].exists():
        print(f'MISSING proba_store: {spec} — skipping')
        continue

    pred = build_pred_frame(
        cfg['proba_path'], cfg['data_path'],
        cfg['regime_col'],  cfg['regime_type'],
    )

    print(f'\n=== {SPEC_LABELS[spec]} | regime: {cfg["regime_col"]} ===')
    cy_counts = pred.drop_duplicates(['country_iso', 'test_year', 'regime'])
    print(pred.dropna(subset=['regime']).groupby('regime')['crisis'].agg(
        crisis_1='sum', total='count'
    ).assign(rate=lambda x: (x['crisis_1'] / x['total']).round(3)))

    for (model, regime), grp in pred.dropna(subset=['regime']).groupby(['model', 'regime']):
        n_pos = int(grp['crisis'].sum())
        n_obs = len(grp)

        if n_pos < MIN_POSITIVES:
            print(f'  SKIP {model}/{regime}: {n_pos} positives (< {MIN_POSITIVES})')
            continue

        auroc_int = roc_auc_score(grp['crisis'], grp['y_proba_integrated'])
        auroc_mac = roc_auc_score(grp['crisis'], grp['y_proba_macro'])
        auroc_mic = roc_auc_score(grp['crisis'], grp['y_proba_micro'])

        h3_records.append({
            'spec':             spec,
            'spec_label':       SPEC_LABELS[spec],
            'regime_col':       cfg['regime_col'],
            'model':            model,
            'regime':           regime,
            'n_obs':            n_obs,
            'n_pos':            n_pos,
            'auroc_integrated': auroc_int,
            'auroc_macro':      auroc_mac,
            'auroc_micro':      auroc_mic,
            # How much do micro features add on top of macro?
            'delta_int_mac':    auroc_int - auroc_mac,
            # How much does the macro context add on top of micro?
            'delta_int_mic':    auroc_int - auroc_mic,
        })

h3_df = pd.DataFrame(h3_records)

print('\n=== H3 summary: mean delta AUROC by regime (averaged across models) ===')
for spec in h3_df['spec'].unique():
    sub = h3_df[h3_df['spec'] == spec]
    tbl = sub.groupby('regime')[['auroc_integrated','auroc_macro','auroc_micro',
                                   'delta_int_mac','delta_int_mic','n_pos']].mean().round(3)
    print(f'\n{SPEC_LABELS[spec]}:')
    print(tbl.to_string())

h3_df.to_parquet(OUTPUT / 'h3_regime_auroc.parquet', index=False)
print('\nSaved → outputs/figures/h3_regime_auroc.parquet')

In [ ]:
# One subplot per spec; bars show delta (integrated − macro) per regime per model.
# A consistent height difference between regimes supports H3.

if not h3_df.empty:
    REGIME_COLORS = {
        'recession': '#d62728',  # red = stress
        'expansion': '#2ca02c',  # green = calm
        'high_gap':  '#f4a582',  # light red — credit gap regime (distinguishable from recession)
        'low_gap':   '#74c476',  # light green — credit gap regime
    }

    specs_list = list(H3_SPECS.keys())
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=False)
    axes = axes.flatten()

    for ax, spec in zip(axes, specs_list):
        sub = h3_df[h3_df['spec'] == spec]
        if sub.empty:
            ax.set_visible(False)
            continue

        regimes      = sorted(sub['regime'].unique())
        models_list  = sorted(sub['model'].unique())
        x     = np.arange(len(models_list))
        width = 0.35

        for i, regime in enumerate(regimes):
            vals = [
                sub[(sub['model'] == m) & (sub['regime'] == regime)]['delta_int_mac'].values
                for m in models_list
            ]
            vals = [v[0] if len(v) > 0 else np.nan for v in vals]
            offset = (i - (len(regimes) - 1) / 2) * width
            ax.bar(x + offset, vals, width,
                   label=regime,
                   color=REGIME_COLORS.get(regime, 'grey'),
                   alpha=0.8, edgecolor='white')

        ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
        ax.set_xticks(x)
        ax.set_xticklabels(models_list, rotation=30, ha='right', fontsize=8)
        regime_var = sub['regime_col'].iloc[0]
        ax.set_title(f'{SPEC_LABELS[spec]}\nregime: {regime_var}', fontsize=10)
        ax.set_ylabel('AUROC(integrated) − AUROC(macro-only)')
        ax.legend(fontsize=8)

    #fig.suptitle(
    #    'H3: Value added by micro features by macro regime\n'
    #    'AUROC(integrated) − AUROC(macro-only), pooled test years 2010–2012',
    #    fontsize=12,
    #)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'h3_regime_delta_auroc.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Second plot: delta_int_mic (value of macro features on top of micro)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=False)
    axes = axes.flatten()

    for ax, spec in zip(axes, specs_list):
        sub = h3_df[h3_df['spec'] == spec]
        if sub.empty:
            ax.set_visible(False)
            continue

        regimes     = sorted(sub['regime'].unique())
        models_list = sorted(sub['model'].unique())
        x     = np.arange(len(models_list))
        width = 0.35

        for i, regime in enumerate(regimes):
            vals = [
                sub[(sub['model'] == m) & (sub['regime'] == regime)]['delta_int_mic'].values
                for m in models_list
            ]
            vals = [v[0] if len(v) > 0 else np.nan for v in vals]
            offset = (i - (len(regimes) - 1) / 2) * width
            ax.bar(x + offset, vals, width,
                   label=regime,
                   color=REGIME_COLORS.get(regime, 'grey'),
                   alpha=0.8, edgecolor='white')

        ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
        ax.set_xticks(x)
        ax.set_xticklabels(models_list, rotation=30, ha='right', fontsize=8)
        regime_var = sub['regime_col'].iloc[0]
        ax.set_title(f'{SPEC_LABELS[spec]}\nregime: {regime_var}', fontsize=10)
        ax.set_ylabel('AUROC(integrated) − AUROC(micro-only)')
        ax.legend(fontsize=8)

    fig.suptitle(
        'H3 (complement): Value added by macro features by macro regime\n'
        'AUROC(integrated) − AUROC(micro-only), pooled test years 2010–2012',
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(OUTPUT / 'h3_regime_delta_auroc_mic.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# H3 regime gap heatmap — one cell per model x spec.
# Gap = calm_delta - stress_delta:
#   GDP specs:  expansion - recession
#   Credit gap: low_gap  - high_gap
# Green = micro more useful in calm conditions; red = stress conditions.
# Near-zero / mixed colours = no clear regime effect.

if not h3_df.empty:
    CALM   = {"gdp": "expansion", "gap": "low_gap"}
    STRESS = {"gdp": "recession",  "gap": "high_gap"}

    models_ordered = sorted(h3_df["model"].unique())
    specs_h3       = list(H3_SPECS.keys())
    col_labels     = [SPEC_LABELS[s] for s in specs_h3]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, delta_col, subtitle in zip(
        axes,
        ["delta_int_mac", "delta_int_mic"],
        ["integrated − macro-only", "integrated − micro-only"],
    ):
        matrix = []
        for model in models_ordered:
            row = []
            for spec in specs_h3:
                sub    = h3_df[(h3_df["spec"] == spec) & (h3_df["model"] == model)]
                rtype  = H3_SPECS[spec]["regime_type"]
                c_vals = sub[sub["regime"] == CALM[rtype]][delta_col].values
                s_vals = sub[sub["regime"] == STRESS[rtype]][delta_col].values
                if len(c_vals) > 0 and len(s_vals) > 0:
                    gap = c_vals[0] - s_vals[0]
                else:
                    gap = np.nan
                row.append(gap)
            matrix.append(row)

        mat  = np.array(matrix, dtype=float)
        vmax = max(np.nanmax(np.abs(mat)), 0.05)

        im = ax.imshow(mat, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")

        ax.set_xticks(range(len(specs_h3)))
        ax.set_xticklabels(col_labels, rotation=20, ha="right", fontsize=9)
        ax.set_yticks(range(len(models_ordered)))
        ax.set_yticklabels(models_ordered, fontsize=9)

        for i in range(len(models_ordered)):
            for j in range(len(specs_h3)):
                val = mat[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:+.2f}", ha="center", va="center",
                            fontsize=8, color="black")

        plt.colorbar(im, ax=ax, shrink=0.85, label="calm − stress gap")
        ax.set_title(subtitle, fontsize=10)

    #fig.suptitle(
    #    "H3 regime gap heatmap\n",
    #    fontsize=10, fontweight="bold",
    #)
    plt.tight_layout()
    plt.savefig(OUTPUT / "h3_regime_gap_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# SECTION 12 ADDITION — H-statistics from NB13 (Friedman H, Reading B evidence)
# Tests whether models capture micro-macro interaction effects.
# H² ≈ 0 = features act independently; H² ≈ 1 = strong interaction.
# Computed for ~28 micro-macro pairs (top-5 × top-5 + 3 theory-driven).
# Covers all model classes — run NB13a/b/c/d first to generate these files.

H_STAT_FILES = {
    '11a_base_lag1':   BASE / 'outputs/attribution/outputs_11a/h_statistic_11a.parquet',
    '11b_robust_lag1': BASE / 'outputs/attribution/outputs_11b/h_statistic_11b.parquet',
    '11c_base_lag2':   BASE / 'outputs/attribution/outputs_11c/h_statistic_11c.parquet',
    '11d_robust_lag2': BASE / 'outputs/attribution/outputs_11d/h_statistic_11d.parquet',
}

for _spec_k, _h_path in H_STAT_FILES.items():
    if not _h_path.exists():
        print(f'H-stat file missing: {_spec_k} — run NB13 first')
        continue

    _h     = pd.read_parquet(_h_path)
    _hmean = _h.groupby(['variant', 'model_name', 'feat1', 'feat2'])['h2'].mean().reset_index()

    for _variant in ['baseline_t1', 'rt1']:
        _sub = _hmean[_hmean['variant'] == _variant]
        if _sub.empty:
            continue

        # Heatmap — micro × macro pairs, averaged across models
        _hagg = _sub.groupby(['feat1', 'feat2'])['h2'].mean().reset_index()
        _hpiv = _hagg.pivot(index='feat1', columns='feat2', values='h2')

        fig, ax = plt.subplots(figsize=(max(8, len(_hpiv.columns) * 1.2),
                                        max(5, len(_hpiv) * 0.55)))
        # Same fixed 0-1 scale as the pooled figure: without vmin/vmax seaborn autoscales
        # to the observed range, which after the centring fix spans only a few thousandths
        # and would paint near-zero interaction as if it were strong.
        sns.heatmap(_hpiv, cmap='YlOrRd', ax=ax, annot=True, fmt='.3f',
                    linewidths=0.4, vmin=0.0, vmax=1.0,
                    cbar_kws={'label': 'H² statistic'})
        ax.set_title(
            f'H-statistics: micro–macro interactions — [NB{_spec_k.split('_')[0]}, {_variant}]\n'
            'Averaged across models and test years  |  H² ≈ 0 = no interaction, H² ≈ 1 = strong',
            fontsize=10
        )
        ax.set_xlabel('Macro feature')
        ax.set_ylabel('Micro feature')
        plt.tight_layout()
        plt.savefig(OUTPUT / f'h_stat_{_spec_k}_{_variant}_agg.png', dpi=150, bbox_inches='tight')
        plt.show()

        # Per-model bar — which model class shows the most interaction?
        _model_mean = _sub.groupby('model_name')['h2'].mean().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(7, 4))
        _model_mean.plot(kind='bar', ax=ax, color='steelblue', alpha=0.85, edgecolor='white')
        ax.set_ylabel('Mean H² (across all pairs and test years)')
        ax.set_title(
            f'Mean interaction strength per model — [NB{_spec_k.split("_")[0]}, {_variant}]',
            fontsize=10
        )
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
        ax.axhline(0.1, color='red', linewidth=0.8, linestyle='--', label='H²=0.1 reference')
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(OUTPUT / f'h_stat_{_spec_k}_{_variant}_per_model.png', dpi=150, bbox_inches='tight')
        plt.show()

        print(f'\n{SPEC_LABELS[_spec_k]} / {_variant} — top interacting pairs:')
        print(_hagg.sort_values('h2', ascending=False).head(8).to_string(index=False))

In [ ]:
# Pooled H-statistic — average H2 across NB11a/b/c/d (baseline_t1)
# Full union of micro-macro pairs; blank cells = pair not computed in any spec.

_H_PATHS = {
    '11a': BASE / 'outputs/attribution/outputs_11a/h_statistic_11a.parquet',
    '11b': BASE / 'outputs/attribution/outputs_11b/h_statistic_11b.parquet',
    '11c': BASE / 'outputs/attribution/outputs_11c/h_statistic_11c.parquet',
    '11d': BASE / 'outputs/attribution/outputs_11d/h_statistic_11d.parquet',
}

_hframes = []
for _spec_k, _hp in _H_PATHS.items():
    if not _hp.exists():
        print(f'Missing: {_hp.name}'); continue
    _hf = pd.read_parquet(_hp)
    _hframes.append(_hf[_hf['variant'] == 'baseline_t1'].copy())

if _hframes:
    _hall   = pd.concat(_hframes, ignore_index=True)
    _hpool  = _hall.groupby(['feat1', 'feat2'])['h2'].mean().reset_index()
    _hpivot = _hpool.pivot(index='feat1', columns='feat2', values='h2')
    _hpivot = _hpivot.reindex(sorted(_hpivot.index)).reindex(sorted(_hpivot.columns), axis=1)

    fig, ax = plt.subplots(figsize=(max(8, len(_hpivot.columns) * 1.3),
                                    max(5, len(_hpivot) * 0.6)))
    # Scale pinned to H's theoretical range 0-1. The previous vmin=0.8 dated from the
    # uncentred computation; corrected H sits at ~0.004, entirely below that floor, so the
    # old scale showed a range the data never enters. Autoscaling instead would stretch
    # differences of a few thousandths across the full palette - the same false impression
    # in a different form. 3 decimals because .2f renders almost every cell as '0.00'.
    sns.heatmap(_hpivot, cmap='YlOrRd', ax=ax, annot=True, fmt='.3f',
                linewidths=0.4, vmin=0.0, vmax=1.0,
                cbar_kws={'label': 'Mean H2'})
    #ax.set_title(
    #    'H-statistics: micro-macro interactions — pooled across specifications (lag-1 & lag-2)\n',
       # 'Mean H2 per pair across all specs, models, test years  |  blank = pair not in any spec\n'
       # 'H2 = 0 no interaction, H2 = 1 strong interaction',
    #    fontsize=10, fontweight='bold',
    #)
    ax.set_xlabel('Macro feature'); ax.set_ylabel('Micro feature')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'h_stat_pooled_baseline_t1.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Coverage: {_hpool.shape[0]} pairs | {_hpivot.shape[0]} micro x {_hpivot.shape[1]} macro')

## 13. Macro-only feature importance (NB12)

Built-in importances from NB12a/b — tree gain and |LR coef|, no SHAP required.
Provides a brief write-up paragraph on which macro indicators drive the macro-only models.

Importances are normalised within each model x test_year window before averaging,
so tree (sum-to-1 gain) and LR (coefficient magnitude) are on a comparable scale.

In [ ]:
FI_PATHS_12 = {
    '12a_macro_base':   BASE / 'outputs/results/12a/fi_store_final.parquet',
    '12b_macro_robust': BASE / 'outputs/results/12b/fi_store_final.parquet',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (spec_key, fi_path) in zip(axes, FI_PATHS_12.items()):
    if not fi_path.exists():
        print(f'MISSING: {fi_path.name}')
        ax.set_visible(False)
        continue

    fi = pd.read_parquet(fi_path)
    fi = fi[fi['variant'] == 'macro_standalone']

    # LR coef_ can be negative; use absolute values so direction does not cancel
    fi['importance'] = fi['importance'].abs()

    # MLP has no native importance (None entries were already excluded at save time)
    fi = fi[fi['model'] != 'mlp']

    # Normalise within each model x test_year so tree gain (sums to 1) and
    # LR |coef| (different magnitude) end up on a comparable 0-1 scale
    fi['imp_norm'] = fi.groupby(['model', 'test_year'])['importance'].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x
    )

    # Mean normalised importance per feature, averaged across models and test years
    mean_imp = fi.groupby('feature')['imp_norm'].mean().sort_values(ascending=True)

    ax.barh(mean_imp.index, mean_imp.values, color='#ff7f0e', alpha=0.85)
    ax.set_xlabel('Mean normalised importance')
    ax.set_title(SPEC_LABELS[spec_key], fontsize=10)

    print(f'{SPEC_LABELS[spec_key]} — feature ranking (mean across models + years):')
    print(mean_imp.sort_values(ascending=False).round(4).to_string())

fig.suptitle(
    'Macro-only feature importance (NB12) — tree gain + |LR coef|, macro_standalone',
    fontsize=12
)
plt.tight_layout()
plt.savefig(OUTPUT / 'macro_fi_nb12.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SECTION 13 ADDITION — macro feature ranking: NB12 (macro-only) vs integrated
# Compares normalised macro feature importance in:
#   (left)  NB12 macro-only models (tree gain + |LR coef|, normalised within model × year)
#   (right) NB11 integrated model (SHAP, macro features only, normalised to macro total)
# Does integration shift which macro features the model relies on?

_SHAP_TO_12 = {'11a': '12a_macro_base', '11b': '12b_macro_robust'}

for _nb_s, _nb_fi in _SHAP_TO_12.items():
    _shap_path = SHAP_FILES.get(_nb_s)
    _fi_path   = FI_PATHS_12.get(_nb_fi)
    if not _shap_path or not _shap_path.exists():
        print(f'SHAP missing: {_nb_s} — skipping')
        continue
    if not _fi_path or not _fi_path.exists():
        print(f'FI missing: {_nb_fi} — skipping')
        continue

    # NB12 macro-only importance (normalised within model × year, averaged across both)
    _fi = pd.read_parquet(_fi_path)
    _fi = _fi[(_fi['variant'] == 'macro_standalone') & (_fi['model'] != 'mlp')].copy()
    _fi['importance'] = _fi['importance'].abs()
    _fi['imp_norm'] = _fi.groupby(['model', 'test_year'])['importance'].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x
    )
    _nb12_rank = _fi.groupby('feature')['imp_norm'].mean()

    # NB11 integrated model — macro SHAP only, normalised to macro total
    _shap = pd.read_parquet(_shap_path)
    _si   = _shap[(_shap['feature_set'] == 'integrated') & (_shap['variant'] == 'baseline_t1')]
    _sc   = [c for c in _si.columns if c.startswith('shap_')]
    _mkc  = [c for c in _sc if c not in micro_shap_cols(_sc)]
    _shap_rank = _si[_mkc].abs().mean()
    _shap_rank.index = [c[5:] for c in _shap_rank.index]
    _shap_norm = _shap_rank / _shap_rank.sum() if _shap_rank.sum() > 0 else _shap_rank

    _shared = sorted(set(_nb12_rank.index) & set(_shap_norm.index))
    if not _shared:
        print(f'No shared macro features: {_nb_s} / {_nb_fi}')
        continue

    _comp = pd.DataFrame({
        'NB12 macro-only': _nb12_rank.reindex(_shared),
        f'NB11{_nb_s[-1]} integrated\n(macro SHAP share)': _shap_norm.reindex(_shared),
    }).fillna(0).sort_values('NB12 macro-only', ascending=True)

    fig, ax = plt.subplots(figsize=(9, max(4, len(_shared) * 0.6)))
    _y = np.arange(len(_comp))
    _w = 0.38
    ax.barh(_y - _w/2, _comp.iloc[:, 0], _w, label='NB12 macro-only',
            color='#ff7f0e', alpha=0.85)
    ax.barh(_y + _w/2, _comp.iloc[:, 1], _w, label=_comp.columns[1],
            color='#2ca02c', alpha=0.85)
    ax.set_yticks(_y)
    ax.set_yticklabels(_comp.index)
    ax.set_xlabel('Mean normalised importance / SHAP share of macro total')
    ax.set_title(
        f'Macro feature importance: NB12 (macro-only) vs NB11{_nb_s[-1]} integrated\n'
        f'{SPEC_LABELS.get(_nb_fi, _nb_fi)}  |  sorted by NB12 ranking',
        fontsize=10
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'macro_fi_comparison_{_nb_s}_vs_{_nb_fi}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n{SPEC_LABELS.get(_nb_fi, _nb_fi)} — macro feature comparison:')
    print(_comp.round(4).to_string())

In [ ]:
# Section 13 delta — macro feature importance shift, original NB12 windows
# delta = NB11 integrated macro SHAP share - NB12 macro-only normalised importance
# Green = gains relative weight when micro added; red = loses weight.

_DELTA_PAIRS = [('11a', '12a_macro_base'), ('11b', '12b_macro_robust')]

for _nb_s, _nb_fi in _DELTA_PAIRS:
    _sp = SHAP_FILES.get(_nb_s); _fp = FI_PATHS_12.get(_nb_fi)
    if not _sp or not _sp.exists() or not _fp or not _fp.exists():
        print(f'Missing: {_nb_s}/{_nb_fi}'); continue

    _fi = pd.read_parquet(_fp)
    _fi = _fi[(_fi['variant'] == 'macro_standalone') & (_fi['model'] != 'mlp')].copy()
    _fi['importance'] = _fi['importance'].abs()
    _fi['imp_norm'] = _fi.groupby(['model', 'test_year'])['importance'].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x)
    _nb12_rank = _fi.groupby('feature')['imp_norm'].mean()

    _shap = pd.read_parquet(_sp)
    _si   = _shap[(_shap['feature_set'] == 'integrated') & (_shap['variant'] == 'baseline_t1')]
    _sc   = [c for c in _si.columns if c.startswith('shap_')]
    _mkc  = [c for c in _sc if c not in micro_shap_cols(_sc)]
    _sr   = _si[_mkc].abs().mean(); _sr.index = [c[5:] for c in _sr.index]
    _snorm = _sr / _sr.sum() if _sr.sum() > 0 else _sr

    _shared = sorted(set(_nb12_rank.index) & set(_snorm.index))
    if not _shared: print(f'No shared features: {_nb_s}/{_nb_fi}'); continue

    _delta = (_snorm.reindex(_shared) - _nb12_rank.reindex(_shared)).sort_values()
    _cols  = ['#2ca02c' if v >= 0 else '#d62728' for v in _delta.values]

    fig, ax = plt.subplots(figsize=(8, max(4, len(_delta) * 0.5)))
    ax.barh(_delta.index, _delta.values, color=_cols, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.9)
    ax.set_xlabel('Delta normalised importance  (NB11 integrated - NB12 macro-only)')
    ax.set_title(
        f'Macro feature importance shift — NB11{_nb_s[-1]} integrated vs NB12 macro-only\n'
        f'Original NB12 windows  |  {SPEC_LABELS.get(_nb_fi, _nb_fi)}',
        fontsize=10)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'macro_fi_delta_{_nb_s}_original.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n{_nb_s} vs {_nb_fi} (original windows):')
    print(_delta.round(4).to_string())

In [ ]:
# Section 13 delta — aligned to NB11 window (2010-2012)
# NB12 fi restricted to test years 2010-2012 so both models evaluated on identical years.
# Removes window difference as confound — remaining delta is from integration only.

_NB11_YEARS = [2010, 2011, 2012]

for _nb_s, _nb_fi in _DELTA_PAIRS:
    _sp = SHAP_FILES.get(_nb_s); _fp = FI_PATHS_12.get(_nb_fi)
    if not _sp or not _sp.exists() or not _fp or not _fp.exists(): continue

    _fi = pd.read_parquet(_fp)
    _fi = _fi[
        (_fi['variant'] == 'macro_standalone') &
        (_fi['model'] != 'mlp') &
        (_fi['test_year'].isin(_NB11_YEARS))
    ].copy()
    if _fi.empty: print(f'No overlapping years for {_nb_fi}'); continue
    _fi['importance'] = _fi['importance'].abs()
    _fi['imp_norm'] = _fi.groupby(['model', 'test_year'])['importance'].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x)
    _nb12_aligned = _fi.groupby('feature')['imp_norm'].mean()

    _shap = pd.read_parquet(_sp)
    _si   = _shap[(_shap['feature_set'] == 'integrated') & (_shap['variant'] == 'baseline_t1')]
    _sc   = [c for c in _si.columns if c.startswith('shap_')]
    _mkc  = [c for c in _sc if c not in micro_shap_cols(_sc)]
    _sr   = _si[_mkc].abs().mean(); _sr.index = [c[5:] for c in _sr.index]
    _snorm = _sr / _sr.sum() if _sr.sum() > 0 else _sr

    _shared = sorted(set(_nb12_aligned.index) & set(_snorm.index))
    if not _shared: continue

    _delta_a = (_snorm.reindex(_shared) - _nb12_aligned.reindex(_shared)).sort_values()
    _cols_a  = ['#2ca02c' if v >= 0 else '#d62728' for v in _delta_a.values]

    fig, ax = plt.subplots(figsize=(8, max(4, len(_delta_a) * 0.5)))
    ax.barh(_delta_a.index, _delta_a.values, color=_cols_a, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.9)
    ax.set_xlabel('Delta normalised importance  (NB11 integrated - NB12 macro-only)')
    ax.set_title(
        f'Macro feature importance shift — NB11{_nb_s[-1]} integrated vs NB12 macro-only\n'
        f'Aligned windows (NB12 restricted to {_NB11_YEARS})  |  {SPEC_LABELS.get(_nb_fi, _nb_fi)}',
        fontsize=10)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'macro_fi_delta_{_nb_s}_aligned.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n{_nb_s} vs {_nb_fi} (aligned 2010-2012):')
    print(_delta_a.round(4).to_string())

## Exploratory AUROC views (presentation candidates)

Per-model specification landscape and the pooled by-year view. The appendix
counterparts live in NB15; these are kept for the presentation appendix.

In [ ]:
# Exploratory AUROC figures (presentation candidates, not thesis appendix)
# Two views kept here rather than in NB15 because they duplicate figures that do
# go in the appendix; these are the per-model / pooled counterparts held for the
# presentation appendix.
#   A  auroc_landscape_per_model  — AUROC across the 8 specifications, one panel
#                                   per model, nothing pooled.
#   C  auroc_byyear_pooled        — AUROC by test year, pooled across models,
#                                   indicator set as line style.
#
# WINDOWS. A reads ACROSS specifications, so it uses the MATCHED values (test
# years 2011-2012). Own-window values would confound the specification with the
# years it happens to be scored on: mean AUROC across models is 0.477 in 2010
# against 0.764 and 0.799 in 2011/2012, and only the lag-1 specs include 2010.
# The two pre-crisis specs are absent because they test 2007 alone (disjoint).
#
# C instead holds the specification fixed and varies the YEAR, so it uses the
# two lag-1 specs, the only ones evaluated on all three years. Indicator set is
# a line style rather than averaged away: extension affects the configurations
# differently, so pooling it would erase Figure 2's finding.
#
# Self-contained on paths and labels ON PURPOSE. NB14 defines its own
# RESULT_FILES pointing at parquets, and label dicts get redefined further down
# the notebook; inheriting either would silently change what is plotted.
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

_B = BASE / 'outputs/results'
_SPECS = [('Baseline\nLag-1',          _B/'11a/results_11a_final.xlsx',        'baseline_t1'),
          ('Extended\nLag-1',          _B/'11b/results_11b_final.xlsx',        'baseline_t1'),
          ('Baseline\nLag-2',          _B/'11c/results_11c_draft4.xlsx',  'baseline_t1'),
          ('Extended\nLag-2',          _B/'11d/results_11d_final.xlsx',   'baseline_t1'),
          ('Onset excl.\nLag-1',       _B/'11a/results_11a_final.xlsx',        'rt1'),
          ('Onset excl. Ext.\nLag-1',  _B/'11b/results_11b_final.xlsx',        'rt1'),
          ('Onset excl.\nLag-2',       _B/'11c/results_11c_draft4.xlsx',  'rt1'),
          ('Onset excl. Ext.\nLag-2',  _B/'11d/results_11d_final.xlsx',   'rt1')]
_YEAR_SETS = [('Baseline indicators', _B/'11a/results_11a_final.xlsx', '-'),
              ('Extended indicators', _B/'11b/results_11b_final.xlsx', '--')]
_MATCH = [2011, 2012]      # the only window shared by all eight specifications
_YEARS = [2010, 2011, 2012]

_MO = ['logistic_regression','decision_tree','random_forest','xgboost','lightgbm','mlp']
_ML = {'logistic_regression':'Logistic Regression','decision_tree':'Decision Tree',
       'random_forest':'Random Forest','xgboost':'XGBoost','lightgbm':'LightGBM','mlp':'MLP-NN'}
_DO = ['micro','macro','integrated']
_DL = {'micro':'Micro','macro':'Macro','integrated':'Integrated'}
_DC = {'micro':'#1f77b4','macro':'#ff7f0e','integrated':'#2ca02c'}

def _load(path, variant, years):
    """Per model x config mean AUROC over `years`. lr_no_dummies is a diagnostic
    model excluded everywhere else in the results, so excluded here too."""
    df = pd.read_excel(path, sheet_name='all_results')
    d = df[(df['variant']==variant) & (df['model']!='lr_no_dummies') & (df['test_year'].isin(years))]
    return d.groupby(['model','dataset'])['auroc'].mean()

# ---- A: specification landscape, one panel per model ------------------------
labs = [s[0] for s in _SPECS]; x = np.arange(len(labs))
Amat = {col: _load(f, v, _MATCH) for col, f, v in _SPECS}

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True)
for ax, mdl in zip(axes.ravel(), _MO):
    for cfg in _DO:
        ax.plot(x, [Amat[c].get((mdl,cfg), np.nan) for c in labs],
                marker='o', ms=5, lw=1.8, color=_DC[cfg], label=_DL[cfg])
    ax.set_title(_ML[mdl], fontsize=11, fontweight='bold')
    ax.axhline(0.5, color='grey', lw=0.8, ls=':')          # no-skill reference
    ax.set_xticks(x); ax.set_xticklabels(labs, rotation=45, ha='right', fontsize=7.5)
    ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True); ax.set_ylim(0.4, 1.02)
for ax in axes[:,0]: ax.set_ylabel('AUROC', fontsize=10)
h,l = axes.ravel()[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, fontsize=10, frameon=False, bbox_to_anchor=(0.5,-0.01))
plt.tight_layout(rect=[0,0.04,1,1])
plt.savefig(OUTPUT/'auroc_landscape_per_model.png', bbox_inches='tight', dpi=200)
plt.show()

# ---- C: by test year, pooled across models ----------------------------------
rows = []
for iset, f, ls in _YEAR_SETS:
    df = pd.read_excel(f, sheet_name='all_results')
    d = df[(df['variant']=='baseline_t1') & (df['model']!='lr_no_dummies') & (df['test_year'].isin(_YEARS))]
    for _, r in d.iterrows():
        rows.append({'iset':iset,'ls':ls,'model':r['model'],'config':r['dataset'],
                     'year':int(r['test_year']),'auroc':r['auroc']})
Y = pd.DataFrame(rows); xy = np.arange(len(_YEARS))

fig, ax = plt.subplots(figsize=(9,6))
for iset, _, ls in _YEAR_SETS:
    for cfg in _DO:
        sub = Y[(Y.iset==iset) & (Y.config==cfg)]
        for mdl in _MO:                                     # faint = the model spread
            s = sub[sub.model==mdl].set_index('year')['auroc'].reindex(_YEARS)
            ax.plot(xy, s.values, color=_DC[cfg], alpha=0.12, lw=0.9, ls=ls, zorder=1)
        m = sub.groupby('year')['auroc'].mean().reindex(_YEARS)
        ax.plot(xy, m.values, marker='o', ms=7, lw=2.8, color=_DC[cfg], ls=ls, zorder=3)
ax.axhline(0.5, color='grey', lw=0.9, ls=':')
ax.set_xticks(xy); ax.set_xticklabels(_YEARS, fontsize=11)
ax.set_xlabel('Test year', fontsize=11); ax.set_ylabel('AUROC', fontsize=11)
ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True); ax.set_xlim(-0.15, len(_YEARS)-0.85)
_h = [Line2D([],[],color=_DC[c],lw=2.8,label=_DL[c]) for c in _DO] + \
     [Line2D([],[],color='grey',lw=2,ls='-',label='Baseline indicators'),
      Line2D([],[],color='grey',lw=2,ls='--',label='Extended indicators')]
ax.legend(handles=_h, fontsize=9, frameon=False, loc='lower right', ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT/'auroc_byyear_pooled.png', bbox_inches='tight', dpi=200)
plt.show()

print('Saved → auroc_landscape_per_model.png, auroc_byyear_pooled.png')
print('\nMean AUROC by year (pooled over models):')
print(Y.groupby(['iset','config','year'])['auroc'].mean().round(3).unstack().to_string())